# 05. Annual k = 12 forecast comparison

This notebook compares annual rolling forecasts for B0--B3. At each forecast origin, eigenmode loadings are recomputed using only the training window.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import binom, betabinom, norm, rankdata, spearmanr, kendalltau
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path(".")
DATA = ROOT / "data"
PDATA = ROOT / "pdata"
FIGURES = ROOT / "figures"
TABLES = ROOT / "tables"
for directory in [DATA, PDATA, FIGURES, TABLES]:
    directory.mkdir(exist_ok=True)

RNG = np.random.default_rng(20260618)
plt.style.use("default")


SECTOR_COLORS = {
    "Basic Materials": "#4C78A8",
    "Capital Goods": "#F58518",
    "Consumer": "#54A24B",
    "Energy": "#E45756",
    "Finance": "#72B7B2",
    "Healthcare": "#B279A2",
    "Technology": "#FF9DA6",
    "Utilities": "#9D755D",
}


def read_sample():
    df = pd.read_csv(DATA / "SAMPLE.csv", parse_dates=["date"])
    expected = ["date", "sector", "obligors", "defaulted"]
    if list(df.columns) != expected:
        raise ValueError(f"Expected columns {expected}, got {list(df.columns)}")
    df = df.sort_values(["date", "sector"]).reset_index(drop=True)
    if (df["obligors"] <= 0).any():
        raise ValueError("obligors must be positive")
    if ((df["defaulted"] < 0) | (df["defaulted"] > df["obligors"])).any():
        raise ValueError("defaulted must lie between zero and obligors")
    return df


def empirical_logit(defaulted, obligors, smooth=0.5):
    defaulted = np.asarray(defaulted, dtype=float)
    obligors = np.asarray(obligors, dtype=float)
    return np.log((defaulted + smooth) / (obligors - defaulted + smooth))


def panel_matrices(df):
    dates = pd.Index(sorted(df["date"].unique()), name="date")
    sectors = pd.Index(sorted(df["sector"].unique()), name="sector")
    dmat = (
        df.pivot(index="date", columns="sector", values="defaulted")
        .reindex(index=dates, columns=sectors)
        .astype(float)
    )
    nmat = (
        df.pivot(index="date", columns="sector", values="obligors")
        .reindex(index=dates, columns=sectors)
        .astype(float)
    )
    x = pd.DataFrame(
        empirical_logit(dmat.to_numpy(), nmat.to_numpy()),
        index=dates,
        columns=sectors,
    )
    rates = dmat / nmat
    return dates, sectors, dmat, nmat, x, rates


def aggregate_k_month(df, k):
    wide_dates = pd.Index(sorted(df["date"].unique()))
    block_lookup = {date: i // k for i, date in enumerate(wide_dates)}
    tmp = df.copy()
    tmp["block"] = tmp["date"].map(block_lookup)
    tmp["block_start"] = tmp["block"].map(lambda b: wide_dates[b * k])
    out = (
        tmp.groupby(["block", "block_start", "sector"], as_index=False)
        .agg(obligors=("obligors", "sum"), defaulted=("defaulted", "sum"))
    )
    out = out.rename(columns={"block_start": "date"})
    out["k_month"] = k
    return out[["date", "sector", "k_month", "obligors", "defaulted"]]


def fit_common_eps_model(df, R=2):
    dates, sectors, dmat, nmat, x, rates = panel_matrices(df)
    X = x.to_numpy()
    intercept = X.mean(axis=0)
    Xc = X - intercept
    cov = np.cov(Xc, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    loadings = eigvecs[:, :R]
    scores = Xc @ loadings
    phis = []
    innov_sd = []
    for r in range(R):
        y = scores[1:, r]
        z = scores[:-1, r]
        denom = float(np.dot(z, z))
        phi = float(np.dot(z, y) / denom) if denom > 1e-12 else 0.0
        phi = float(np.clip(phi, -0.98, 0.98))
        resid = y - phi * z
        phis.append(phi)
        innov_sd.append(float(np.sqrt(np.mean(resid**2) + 1e-8)))
    fitted_centered = scores @ loadings.T
    residual = Xc - fitted_centered
    sigma_eps_common = float(np.sqrt(np.mean(residual**2) + 1e-8))
    eta_hat = intercept + fitted_centered
    p_hat = expit(eta_hat)
    ll = float(binom.logpmf(dmat.to_numpy(), nmat.to_numpy(), p_hat).sum())
    n_params = len(sectors) + R * (len(sectors) + 2) + 1
    aic = float(2 * n_params - 2 * ll)
    return {
        "dates": list(dates),
        "sectors": list(sectors),
        "defaulted": dmat,
        "obligors": nmat,
        "logit": x,
        "rates": rates,
        "intercept": intercept,
        "eigvals": eigvals,
        "loadings": loadings,
        "scores": scores,
        "phis": np.array(phis),
        "innov_sd": np.array(innov_sd),
        "sigma_eps_common": sigma_eps_common,
        "eta_hat": eta_hat,
        "p_hat": p_hat,
        "log_likelihood": ll,
        "aic": aic,
    }


def acf_values(x, max_lag=24):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    denom = np.dot(x, x)
    if denom <= 1e-12:
        return np.zeros(max_lag + 1)
    vals = [1.0]
    for lag in range(1, max_lag + 1):
        vals.append(float(np.dot(x[:-lag], x[lag:]) / denom))
    return np.array(vals)


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def beta_binomial_params(rates, obligors, defaulted):
    total_d = float(np.sum(defaulted))
    total_n = float(np.sum(obligors))
    mu = np.clip(total_d / total_n, 1e-6, 1 - 1e-6)
    r = np.asarray(rates, dtype=float)
    v = float(np.nanvar(r, ddof=1)) if len(r) > 1 else 0.0
    base = mu * (1 - mu) / max(float(np.nanmean(obligors)), 1.0)
    extra = max(v - base, 0.0)
    if extra <= 1e-12:
        kappa = 1e5
    else:
        kappa = max(mu * (1 - mu) / extra - 1.0, 2.0)
    return mu * kappa, (1.0 - mu) * kappa


def simulate_dynamic_counts(model, future_obligors, R, n_draws=300, seed=20260618):
    rng = np.random.default_rng(seed)
    sectors = list(model["sectors"])
    H = future_obligors.shape[0]
    S = future_obligors.shape[1]
    last_score = model["scores"][-1, :R].copy()
    loadings = model["loadings"][:, :R]
    draws = np.zeros((n_draws, H, S), dtype=float)
    for m in range(n_draws):
        score = last_score.copy()
        for h in range(H):
            for r in range(R):
                score[r] = model["phis"][r] * score[r] + rng.normal(0.0, model["innov_sd"][r])
            eps = rng.normal(0.0, model["sigma_eps_common"], size=S)
            eta = model["intercept"] + score @ loadings.T + eps
            p = np.clip(expit(eta), 1e-8, 1 - 1e-8)
            draws[m, h, :] = rng.binomial(future_obligors[h, :].astype(int), p)
    return draws

In [ ]:
df = read_sample()
dates = sorted(df["date"].unique())
min_train = 120
horizon = 12
step = 6
origins = list(range(min_train, len(dates) - horizon + 1, step))

records = []
for origin_idx in origins:
    train_dates = dates[:origin_idx]
    test_dates = dates[origin_idx:origin_idx + horizon]
    train = df[df["date"].isin(train_dates)].copy()
    test = df[df["date"].isin(test_dates)].copy()
    _, sectors, dtest_monthly, ntest_monthly, _, _ = panel_matrices(test)
    annual_d = dtest_monthly.sum(axis=0)
    annual_n = ntest_monthly.sum(axis=0)
    origin_label = pd.Timestamp(dates[origin_idx]).strftime("%Y-%m-%d")

    train_panel = train.assign(default_rate=train["defaulted"] / train["obligors"])
    for sector in sectors:
        g = train_panel[train_panel["sector"] == sector]
        observed_d = int(annual_d[sector])
        observed_n = int(annual_n[sector])
        observed_rate = observed_d / observed_n

        p0 = np.clip(g["defaulted"].sum() / g["obligors"].sum(), 1e-8, 1 - 1e-8)
        records.append({
            "origin": origin_label,
            "sector": sector,
            "model": "B0 = Binomial-best static baseline",
            "observed_defaulted": observed_d,
            "observed_obligors": observed_n,
            "observed_rate": observed_rate,
            "predicted_rate": p0,
            "log_score": float(binom.logpmf(observed_d, observed_n, p0)),
        })

        alpha, beta = beta_binomial_params(g["default_rate"], g["obligors"], g["defaulted"])
        p1 = alpha / (alpha + beta)
        records.append({
            "origin": origin_label,
            "sector": sector,
            "model": "B1 = Beta-Binomial-best static baseline",
            "observed_defaulted": observed_d,
            "observed_obligors": observed_n,
            "observed_rate": observed_rate,
            "predicted_rate": p1,
            "log_score": float(betabinom.logpmf(observed_d, observed_n, alpha, beta)),
        })

    # No-look-ahead rule: fixed eigenmode loadings are recomputed here
    # from the training window only at each rolling forecast origin.
    for R, label in [
        (1, "B2 = One-factor dynamic common-eps model"),
        (2, "B3 = Two-factor dynamic common-eps model"),
    ]:
        model = fit_common_eps_model(train, R=R)
        future_n = ntest_monthly.to_numpy()
        draws = simulate_dynamic_counts(model, future_n, R=R, n_draws=300, seed=20260618 + origin_idx + R)
        annual_draws = draws.sum(axis=1)
        mean_counts = annual_draws.mean(axis=0)
        for s, sector in enumerate(model["sectors"]):
            observed_d = int(annual_d[sector])
            observed_n = int(annual_n[sector])
            observed_rate = observed_d / observed_n
            pred_rate = float(np.clip(mean_counts[s] / observed_n, 1e-8, 1 - 1e-8))
            records.append({
                "origin": origin_label,
                "sector": sector,
                "model": label,
                "observed_defaulted": observed_d,
                "observed_obligors": observed_n,
                "observed_rate": observed_rate,
                "predicted_rate": pred_rate,
                "log_score": float(binom.logpmf(observed_d, observed_n, pred_rate)),
            })

forecast = pd.DataFrame(records)
forecast.to_csv(PDATA / "annual_forecast_scores.csv", index=False)

table6 = (
    forecast.groupby(["model", "sector"], as_index=False)
    .agg(
        mean_log_score=("log_score", "mean"),
        rmse_rate=("predicted_rate", lambda x: np.sqrt(np.mean((x - forecast.loc[x.index, "observed_rate"]) ** 2))),
        n_forecasts=("log_score", "size"),
    )
)
table6.to_csv(TABLES / "table6_annual_forecast_scores.csv", index=False)

table7 = (
    forecast.groupby("model")
    .apply(lambda g: pd.Series({
        "mean_log_score": g["log_score"].mean(),
        "rmse_rate": np.sqrt(np.mean((g["observed_rate"] - g["predicted_rate"]) ** 2)),
        "mean_predicted_rate": g["predicted_rate"].mean(),
        "mean_observed_rate": g["observed_rate"].mean(),
        "n_forecasts": len(g),
    }))
    .reset_index()
)
order = [
    "B0 = Binomial-best static baseline",
    "B1 = Beta-Binomial-best static baseline",
    "B2 = One-factor dynamic common-eps model",
    "B3 = Two-factor dynamic common-eps model",
]
table7["model"] = pd.Categorical(table7["model"], categories=order, ordered=True)
table7 = table7.sort_values("model")
table7.to_csv(TABLES / "table7_forecast_comparison.csv", index=False)
table7

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
colors = ["#4C78A8", "#F58518", "#54A24B", "#E45756"]
axes[0].bar(table7["model"].astype(str), table7["mean_log_score"], color=colors)
axes[0].set_title("Mean annual log score")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylabel("Higher is better")

axes[1].bar(table7["model"].astype(str), table7["rmse_rate"], color=colors)
axes[1].set_title("Annual default-rate RMSE")
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_ylabel("Lower is better")
fig.suptitle("Figure 8. Annual k=12 forecast scores")
savefig(FIGURES / "figure8_forecast_scores.png")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.2))
for model_name, color in zip(table7["model"].astype(str), colors):
    g = forecast[forecast["model"].astype(str) == model_name]
    ax.scatter(g["observed_rate"], g["predicted_rate"], s=20, alpha=0.55, label=model_name.split(" = ")[0], color=color)
lim = [0, max(forecast["observed_rate"].max(), forecast["predicted_rate"].max()) * 1.05]
ax.plot(lim, lim, color="black", lw=1)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("Observed annual default rate")
ax.set_ylabel("Predicted annual default rate")
ax.set_title("Figure 9. Forecast calibration")
ax.legend(fontsize=8)
savefig(FIGURES / "figure9_forecast_calibration.png")

In [ ]:
cum = (
    forecast.groupby(["origin", "model"], as_index=False)["log_score"].sum()
    .sort_values(["model", "origin"])
)
cum["cumulative_log_score"] = cum.groupby("model")["log_score"].cumsum()

fig, ax = plt.subplots(figsize=(9, 4.8))
for model_name, color in zip(table7["model"].astype(str), colors):
    g = cum[cum["model"].astype(str) == model_name]
    ax.plot(pd.to_datetime(g["origin"]), g["cumulative_log_score"], lw=1.4, label=model_name.split(" = ")[0], color=color)
ax.set_xlabel("Forecast origin")
ax.set_ylabel("Cumulative annual log score")
ax.set_title("Figure 11. Cumulative forecast log score")
ax.legend(ncol=2, fontsize=8)
savefig(FIGURES / "figure11_cumulative_forecast_log_score.png")